In [165]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

### Exploratory data analysis (EDA)

In [166]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"

train_path = DATA_DIR / "final_proj_data.csv"
test_path = DATA_DIR / "final_proj_test.csv"

df_train = pd.read_csv("../data/final_proj_data.csv")
df_test = pd.read_csv("../data/final_proj_test.csv")

In [167]:
df_train.sample(10)

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var222,Var223,Var224,Var225,Var226,Var227,Var228,Var229,Var230,y
6855,NaN,NaN,NaN,NaN,NaN,441.0,7.0,NaN,NaN,NaN,...,catzS2D,jySVZNlOJy,NaN,ELof,Qcbd,ZI9m,ib5G6X1eUxUn6,mj86,NaN,0
3025,NaN,NaN,NaN,NaN,NaN,1449.0,7.0,NaN,NaN,NaN,...,j32r3XO,LM8l689qOp,NaN,kG3k,me1d,RAYp,F2FyR07IdsN7I,am7c,NaN,0
7511,NaN,NaN,NaN,NaN,NaN,168.0,7.0,NaN,NaN,NaN,...,IIvC99a,LM8l689qOp,NaN,ELof,PM2D,RAYp,F2FyR07IdsN7I,am7c,NaN,0
4258,NaN,NaN,NaN,NaN,935140.0,NaN,NaN,NaN,NaN,0.0,...,kWFEVJ2,LM8l689qOp,NaN,NaN,7P5s,RAYp,F2FyR07IdsN7I,NaN,NaN,0
4417,NaN,NaN,NaN,NaN,NaN,294.0,0.0,NaN,NaN,NaN,...,XlgxB9z,LM8l689qOp,NaN,NaN,FSa2,RAYp,F2FyR07IdsN7I,NaN,NaN,1
7618,NaN,NaN,NaN,NaN,NaN,301.0,7.0,NaN,NaN,NaN,...,catzS2D,LM8l689qOp,NaN,ELof,Qu4f,ZI9m,ib5G6X1eUxUn6,am7c,NaN,0
6067,NaN,NaN,NaN,NaN,NaN,525.0,7.0,NaN,NaN,NaN,...,TX2AGfT,NaN,NaN,NaN,kwS7,RAYp,F2FyR07IdsN7I,NaN,NaN,0
9571,NaN,0.0,327.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,FS4qjNq,LM8l689qOp,4n2X,NaN,WqMG,RAYp,F2FyR07IdsN7I,NaN,NaN,0
1557,NaN,NaN,NaN,NaN,NaN,847.0,7.0,NaN,NaN,NaN,...,rJxz_Kt,LM8l689qOp,NaN,kG3k,FSa2,RAYp,F2FyR07IdsN7I,am7c,NaN,0
3700,NaN,NaN,NaN,NaN,NaN,1029.0,7.0,NaN,NaN,NaN,...,jrbDSFl,jySVZNlOJy,NaN,ELof,Xa3G,RAYp,F2FyR07IdsN7I,mj86,NaN,0


In [168]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 231 entries, Var1 to y
dtypes: float64(191), int64(2), object(38)
memory usage: 17.6+ MB


In [169]:
df_train.describe()

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var184,Var185,Var186,Var187,Var188,Var189,Var190,Var209,Var230,y
count,133.000000,266.0,266.000000,280.000000,2.410000e+02,8980.000000,8995.000000,0.0,133.000000,2.410000e+02,...,266.000000,0.0,133.000000,133.000000,266.000000,4206.000000,43.000000,0.0,0.0,10000.00000
mean,14.977444,0.0,341.052632,0.096429,2.338101e+05,1340.916258,6.860700,NaN,61.383459,3.672943e+05,...,6.180451,NaN,2.977444,20.601504,159.107368,272.455064,25725.112326,NaN,NaN,0.13050
std,66.456008,0.0,2810.606975,0.928243,5.532305e+05,2380.516758,6.300994,NaN,266.124849,8.234215e+05,...,12.177204,NaN,10.329764,93.736247,115.766972,86.752531,37487.484852,NaN,NaN,0.33687
min,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,NaN,0.000000,0.000000e+00,...,0.000000,NaN,0.000000,0.000000,0.000000,6.000000,0.000000,NaN,NaN,0.00000
25%,0.000000,0.0,0.000000,0.000000,0.000000e+00,523.250000,0.000000,NaN,2.000000,0.000000e+00,...,0.000000,NaN,0.000000,0.000000,18.840000,204.000000,1312.875000,NaN,NaN,0.00000
50%,0.000000,0.0,0.000000,0.000000,0.000000e+00,861.000000,7.000000,NaN,18.000000,0.000000e+00,...,0.000000,NaN,0.000000,4.000000,194.670000,270.000000,10853.820000,NaN,NaN,0.00000
75%,16.000000,0.0,0.000000,0.000000,1.172350e+05,1428.000000,7.000000,NaN,40.000000,2.439360e+05,...,7.000000,NaN,0.000000,12.000000,247.080000,330.000000,37491.525000,NaN,NaN,0.00000
max,680.000000,0.0,42588.000000,9.000000,3.024000e+06,76195.000000,35.000000,NaN,2300.000000,6.394806e+06,...,64.000000,NaN,102.000000,878.000000,452.760000,642.000000,191167.200000,NaN,NaN,1.00000


In [170]:
y = df_train["y"]
X = df_train.drop(columns=["y"])

y.value_counts(normalize=True)
# sns.countplot(x=y)

y
0    0.8695
1    0.1305
Name: proportion, dtype: float64

In [171]:
# Find and delete empty column
cals_all_nan_train = X.columns[X.isna().all()]
cals_all_nan_test = df_test.columns[df_test.isna().all()]

cals_to_drop = cals_all_nan_train.intersection(cals_all_nan_test)

X = X.drop(columns=cals_to_drop)
df_test = df_test.drop(columns=cals_to_drop)

In [172]:
print(X.shape)
print(df_test.shape)

(10000, 212)
(2500, 212)


In [173]:
X.describe()

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var9,Var10,Var11,...,Var180,Var181,Var182,Var183,Var184,Var186,Var187,Var188,Var189,Var190
count,133.000000,266.0,266.000000,280.000000,2.410000e+02,8980.000000,8995.000000,133.000000,2.410000e+02,266.000000,...,1.330000e+02,9080.000000,2.800000e+02,2.660000e+02,266.000000,133.000000,133.000000,266.000000,4206.000000,43.000000
mean,14.977444,0.0,341.052632,0.096429,2.338101e+05,1340.916258,6.860700,61.383459,3.672943e+05,8.421053,...,3.727323e+06,0.586674,1.333466e+06,6.326152e+04,6.180451,2.977444,20.601504,159.107368,272.455064,25725.112326
std,66.456008,0.0,2810.606975,0.928243,5.532305e+05,2380.516758,6.300994,266.124849,8.234215e+05,2.156904,...,3.709497e+06,2.470885,2.227988e+06,1.623765e+05,12.177204,10.329764,93.736247,115.766972,86.752531,37487.484852
min,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,8.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,6.000000,0.000000
25%,0.000000,0.0,0.000000,0.000000,0.000000e+00,523.250000,0.000000,2.000000,0.000000e+00,8.000000,...,2.554440e+05,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,18.840000,204.000000,1312.875000
50%,0.000000,0.0,0.000000,0.000000,0.000000e+00,861.000000,7.000000,18.000000,0.000000e+00,8.000000,...,2.452807e+06,0.000000,2.834700e+04,0.000000e+00,0.000000,0.000000,4.000000,194.670000,270.000000,10853.820000
75%,16.000000,0.0,0.000000,0.000000,1.172350e+05,1428.000000,7.000000,40.000000,2.439360e+05,8.000000,...,6.362363e+06,0.000000,1.463264e+06,3.585300e+04,7.000000,0.000000,12.000000,247.080000,330.000000,37491.525000
max,680.000000,0.0,42588.000000,9.000000,3.024000e+06,76195.000000,35.000000,2300.000000,6.394806e+06,32.000000,...,1.327067e+07,49.000000,1.052112e+07,1.209600e+06,64.000000,102.000000,878.000000,452.760000,642.000000,191167.200000


In [174]:
n_rows = X.shape[0]
non_null_counts = X.notna().sum().sort_values()
non_null_ratio = non_null_counts / n_rows
# plt.figure(figsize=(10, 6))
# sns.histplot(non_null_ratios, bins=50, kde=False)
non_null_counts.head(10)
non_null_ratio.head(10)

Var118    0.0043
Var92     0.0043
Var190    0.0043
Var64     0.0046
Var45     0.0078
Var102    0.0082
Var12     0.0104
Var98     0.0104
Var62     0.0104
Var56     0.0131
dtype: float64

In [175]:
threshold = 0.1 # 10%

cols_very_sparse = non_null_ratio[non_null_ratio < threshold].index
len(cols_very_sparse), cols_very_sparse[:10]


(136,
 Index(['Var118', 'Var92', 'Var190', 'Var64', 'Var45', 'Var102', 'Var12',
        'Var98', 'Var62', 'Var56'],
       dtype='object'))

In [176]:
X = X.drop(columns=cols_very_sparse)
df_test = df_test.drop(columns=cols_very_sparse)
X.shape, df_test.shape

((10000, 76), (2500, 76))

In [177]:
num_cols = X.select_dtypes(include=["float64", "int64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns
X_num = X[num_cols].copy()
X_cat = X[cat_cols].copy()

X_cat.nunique().sort_values()

Var201       2
Var218       2
Var211       2
Var208       2
Var225       3
Var194       3
Var205       3
Var196       3
Var203       4
Var223       4
Var229       4
Var210       6
Var221       7
Var227       7
Var207      12
Var219      17
Var195      18
Var206      21
Var226      23
Var228      29
Var193      40
Var212      65
Var204     100
Var197     185
Var192     297
Var216     977
Var199    1850
Var222    2100
Var198    2100
Var220    2100
Var202    3802
Var200    4478
Var214    4478
Var217    5529
dtype: int64

In [178]:
"""I'm not sure, but the columns (3802, 4478, 5529, with 10000 rows) seem to be: 
Customer ID/contract number/ some kind of hash / almost unique token. 
Such columns often do not help the model much, and sometimes even harm it"""

nunique_cat = X_cat.nunique()
cols_high_cardinality = nunique_cat[nunique_cat > 1000].index
print(len(cols_high_cardinality), cols_high_cardinality.tolist()[:10])
X = X.drop(columns=cols_high_cardinality)
df_test = df_test.drop(columns=cols_high_cardinality)
X.shape, df_test.shape

8 ['Var198', 'Var199', 'Var200', 'Var202', 'Var214', 'Var217', 'Var220', 'Var222']


((10000, 68), (2500, 68))

In [179]:
# Record numerical and categorical features

num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")


Numerical columns: 42
Categorical columns: 26


In [180]:
missing_num = X[num_cols].isna().mean().sort_values(ascending=False)
missing_cat = X[cat_cols].isna().mean().sort_values(ascending=False)

print(f"Missinf in numerical columns: {missing_num.head(10)}")
print(f"Missinf in categorical columns: {missing_cat.head(10)}")

Missinf in numerical columns: Var189    0.5794
Var72     0.4386
Var94     0.4386
Var126    0.2780
Var149    0.1360
Var109    0.1360
Var24     0.1360
Var81     0.1020
Var21     0.1020
Var6      0.1020
dtype: float64
Missinf in categorical columns: Var201    0.7467
Var194    0.7467
Var229    0.5561
Var225    0.5109
Var206    0.1020
Var223    0.1013
Var219    0.1013
Var205    0.0386
Var218    0.0128
Var192    0.0079
dtype: float64


### Model training trial

In [181]:
X_num_imp = X[num_cols].copy()
X_num_imp = X_num_imp.fillna(X_num_imp.median())

# Categorical: NaN -> 'Missing'
X_cat_imp = X[cat_cols].copy()
X_cat_imp = X_cat_imp.fillna("Missing")

# One-Hot for categorical features
X_cat_dum = pd.get_dummies(X_cat_imp, drop_first=False)

X_prepared = pd.concat([X_num_imp, X_cat_dum], axis=1)

X_prepared.shape


(10000, 1891)

In [182]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    rf_clf,
    X_prepared,
    y,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1
)

scores, scores.mean(), scores.std()


(array([0.60747027, 0.58802566, 0.62471165, 0.61953957, 0.60191373]),
 np.float64(0.6083321766373857),
 np.float64(0.013024490292493175))

In [183]:
# 1. Calculate MI for all features with X_prepared
mi_scores = mutual_info_classif(
    X_prepared,
    y,
    discrete_features='auto',   # sklearn will decide what to consider discrete
    random_state=42
)

mi = pd.Series(mi_scores, index=X_prepared.columns).sort_values(ascending=False)

# 2. Here are the top 20 most informative signs
mi.head(20)

Var126                        0.081412
Var73                         0.065427
Var212_NhsEn4L                0.052956
Var228_F2FyR07IdsN7I          0.038999
Var193_RO12                   0.032076
Var13                         0.031756
Var227_RAYp                   0.028041
Var74                         0.024019
Var193_2Knk1KF                0.023976
Var7                          0.022328
Var229_Missing                0.022022
Var207_me75fM6ugJ             0.021417
Var225_Missing                0.021272
Var140                        0.020573
Var212_XfqtO3UdzaXh_          0.020550
Var221_oslk                   0.020318
Var228_55YFVY9                0.019926
Var189                        0.016578
Var125                        0.016167
Var207_7M47J5GA0pTYIFxg5uy    0.015652
dtype: float64

### Manual version Preprocessing + Pipeline

In [184]:
# 1. Numeric columns
num_cols = X.select_dtypes(include=[np.number]).columns

# 2. Categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns

# 3. Verification.
print(f"Numeric columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print("The first numerical:", list(num_cols[:5]))
print("The first categorical:", list(cat_cols[:5]))


Numeric columns: 42
Categorical columns: 26
The first numerical: ['Var6', 'Var7', 'Var13', 'Var21', 'Var22']
The first categorical: ['Var192', 'Var193', 'Var194', 'Var195', 'Var196']


In [185]:
numeric_imputer = SimpleImputer(strategy="median")
numeric_imputer.fit(X[num_cols])
X_num_imputed = numeric_imputer.transform(X[num_cols])
categorical_imputer = SimpleImputer(strategy="most_frequent")
categorical_imputer.fit(X[cat_cols])
X_cat_imputed = categorical_imputer.transform(X[cat_cols])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat_ohe = ohe.fit_transform(X_cat_imputed)
X_cat_ohe.shape
 

(10000, 1836)

### Automatic mode Preprocessing + Pipeline

In [186]:
numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, num_cols),
                                               ("cat", categorical_transformer, cat_cols)
                                ]
)

In [187]:
rm_clf = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rm_clf)
])

In [188]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model_pipeline,
    X, y,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1
)

scores, scores.mean(), scores.std()

(array([0.59951992, 0.59980744, 0.62825555, 0.60833284, 0.61312046]),
 np.float64(0.6098072393743708),
 np.float64(0.01057654413352732))